# Required ML Reporting Contract Gate

Fails closed before Reporting publication when a required ML Delta table is absent, empty, incompatible, temporally incomplete, or violates its grain and value constraints. It never creates or repairs output tables.


In [ ]:
from functools import reduce

from pyspark.sql import functions as F
from pyspark.sql.utils import AnalysisException


In [ ]:
# Required Reporting contract. Repository validation keeps this in exact agreement
# with the four producer declarations, the solution manifest, and active TMDL.
LAKEHOUSE_NAME = "retail_lakehouse"
GOLD_DB = "au"

ML_OUTPUT_CONTRACTS = {'demand_forecast': [('store_id', 'long', False),
                     ('product_id', 'long', False),
                     ('forecast_date', 'date', False),
                     ('predicted_units', 'double', False),
                     ('lower_bound', 'double', False),
                     ('upper_bound', 'double', False),
                     ('mape', 'double', False),
                     ('source_as_of', 'timestamp', False),
                     ('generated_at', 'timestamp', False),
                     ('forecast_horizon_days', 'int', False),
                     ('model_run_id', 'string', False),
                     ('schema_version', 'string', False)],
 'customer_segments': [('customer_id', 'long', False),
                       ('cluster_id', 'int', False),
                       ('segment_label', 'string', False),
                       ('recency_days', 'double', False),
                       ('frequency', 'double', False),
                       ('monetary_value', 'double', False),
                       ('avg_order_value', 'double', False),
                       ('first_purchase_date', 'date', False),
                       ('last_purchase_date', 'date', False),
                       ('segmented_at', 'timestamp', False),
                       ('generated_at', 'timestamp', False),
                       ('model_run_id', 'string', False),
                       ('schema_version', 'string', False)],
 'churn_predictions': [('customer_id', 'long', False),
                       ('churn_probability', 'double', False),
                       ('churn_prediction', 'int', False),
                       ('risk_category', 'string', False),
                       ('is_churned_actual', 'int', True),
                       ('prediction_date', 'timestamp', False),
                       ('generated_at', 'timestamp', False),
                       ('model_version', 'string', False),
                       ('churn_window_days', 'int', False),
                       ('model_run_id', 'string', False),
                       ('schema_version', 'string', False)],
 'stockout_risk': [('store_id', 'long', False),
                   ('product_id', 'long', False),
                   ('inventory_as_of', 'timestamp', False),
                   ('current_inventory', 'long', False),
                   ('demand_velocity_daily', 'double', False),
                   ('days_of_inventory', 'double', False),
                   ('demand_trend', 'double', False),
                   ('stockout_probability', 'double', False),
                   ('stockout_predicted', 'int', False),
                   ('risk', 'string', False),
                   ('ranking', 'int', False),
                   ('risk_level', 'string', False),
                   ('predicted_at', 'timestamp', False),
                   ('generated_at', 'timestamp', False),
                   ('forecast_horizon_days', 'int', False),
                   ('Department', 'string', True),
                   ('Category', 'string', True),
                   ('Subcategory', 'string', True),
                   ('model_run_id', 'string', False),
                   ('schema_version', 'string', False)]}

REQUIRED_ML_RULES = {'demand_forecast': {'grain': ('store_id', 'product_id', 'forecast_date'),
                     'as_of': 'generated_at',
                     'lineage': ('source_as_of', 'model_run_id', 'schema_version'),
                     'probabilities': (),
                     'horizon': 'forecast_horizon_days'},
 'customer_segments': {'grain': ('customer_id',),
                       'as_of': 'generated_at',
                       'lineage': ('segmented_at', 'model_run_id', 'schema_version'),
                       'probabilities': (),
                       'horizon': None},
 'churn_predictions': {'grain': ('customer_id',),
                       'as_of': 'generated_at',
                       'lineage': ('prediction_date', 'model_run_id', 'model_version', 'schema_version'),
                       'probabilities': ('churn_probability',),
                       'horizon': None},
 'stockout_risk': {'grain': ('store_id', 'product_id', 'forecast_horizon_days'),
                   'as_of': 'generated_at',
                   'lineage': ('inventory_as_of', 'predicted_at', 'model_run_id', 'schema_version'),
                   'probabilities': ('stockout_probability',),
                   'horizon': None}}


In [ ]:
def _normalize_ml_type(data_type):
    return data_type.replace("bigint", "long").replace("integer", "int")


def _fail_if(frame, condition, message):
    if frame.filter(condition).limit(1).count():
        raise RuntimeError(message)


def _any_null(columns):
    return reduce(lambda left, right: left | right, (F.col(name).isNull() for name in columns))


def _any_blank(columns):
    return reduce(
        lambda left, right: left | right,
        (F.trim(F.col(name).cast("string")) == "" for name in columns),
    )


def _any_non_finite(columns):
    return reduce(
        lambda left, right: left | right,
        (
            F.isnan(F.col(name))
            | F.col(name).isin(float("inf"), float("-inf"))
            for name in columns
        ),
    )


def _validate_schema(frame, table_name):
    expected = tuple((name, data_type) for name, data_type, _ in ML_OUTPUT_CONTRACTS[table_name])
    actual = tuple(
        (field.name, _normalize_ml_type(field.dataType.simpleString()))
        for field in frame.schema.fields
    )
    if actual != expected:
        raise RuntimeError(
            f"Required ML schema mismatch for {table_name}: expected={expected}, actual={actual}"
        )


def _validate_demand_forecast(frame):
    _fail_if(
        frame,
        (F.col("predicted_units") < 0)
        | (F.col("lower_bound") < 0)
        | (F.col("upper_bound") < F.col("lower_bound"))
        | (F.col("predicted_units") < F.col("lower_bound"))
        | (F.col("predicted_units") > F.col("upper_bound"))
        | (F.col("mape") < 0)
        | (F.col("forecast_horizon_days") <= 0)
        | (F.col("source_as_of") > F.col("generated_at")),
        "demand_forecast contains invalid forecast bounds or horizon values",
    )
    stepped = frame.withColumn(
        "_horizon_step",
        F.datediff("forecast_date", F.to_date("source_as_of")),
    )
    _fail_if(
        stepped,
        (F.col("_horizon_step") < 1)
        | (F.col("_horizon_step") > F.col("forecast_horizon_days")),
        "demand_forecast contains dates outside its declared horizon",
    )
    horizon = stepped.groupBy("store_id", "product_id").agg(
        F.countDistinct("forecast_date").alias("date_count"),
        F.countDistinct("forecast_horizon_days").alias("horizon_count"),
        F.min("_horizon_step").alias("min_step"),
        F.max("_horizon_step").alias("max_step"),
        F.max("forecast_horizon_days").alias("horizon_days"),
    )
    _fail_if(
        horizon,
        (F.col("horizon_count") != 1)
        | (F.col("date_count") != F.col("horizon_days"))
        | (F.col("min_step") != 1)
        | (F.col("max_step") != F.col("horizon_days")),
        "demand_forecast has an incomplete or inconsistent forecast horizon",
    )


def _validate_customer_segments(frame):
    _fail_if(
        frame,
        (F.col("cluster_id") < 0)
        | (F.col("recency_days") < 0)
        | (F.col("frequency") <= 0)
        | (F.col("monetary_value") <= 0)
        | (F.col("avg_order_value") <= 0)
        | (F.col("first_purchase_date") > F.col("last_purchase_date"))
        | (F.col("segmented_at") > F.col("generated_at")),
        "customer_segments contains invalid RFM values or purchase dates",
    )


def _validate_churn_predictions(frame):
    _fail_if(
        frame,
        (~F.col("churn_prediction").isin(0, 1))
        | F.col("is_churned_actual").isNotNull()
        | (F.col("churn_window_days") <= 0)
        | (F.col("prediction_date") > F.col("generated_at")),
        "churn_predictions contains an invalid class, compatibility value, or horizon",
    )


def _validate_stockout_risk(frame):
    _fail_if(
        frame,
        (~F.col("stockout_predicted").isin(0, 1))
        | (F.col("current_inventory") < 0)
        | (F.col("demand_velocity_daily") < 0)
        | (F.col("days_of_inventory") < 0)
        | (F.col("forecast_horizon_days") <= 0)
        | (F.col("ranking") <= 0)
        | (F.col("inventory_as_of") > F.col("predicted_at"))
        | (F.col("predicted_at") > F.col("generated_at")),
        "stockout_risk contains invalid inventory, class, horizon, or as-of values",
    )


TABLE_VALIDATORS = {
    "demand_forecast": _validate_demand_forecast,
    "customer_segments": _validate_customer_segments,
    "churn_predictions": _validate_churn_predictions,
    "stockout_risk": _validate_stockout_risk,
}


def validate_required_table(table_name):
    full_name = f"{LAKEHOUSE_NAME}.{GOLD_DB}.{table_name}"
    try:
        frame = spark.table(full_name)
    except AnalysisException as exc:
        raise RuntimeError(f"Required ML table is missing: {full_name}") from exc
    if frame.limit(1).count() == 0:
        raise RuntimeError(f"Required ML table is empty: {full_name}")

    _validate_schema(frame, table_name)
    rule = REQUIRED_ML_RULES[table_name]
    required_values = (*rule["grain"], rule["as_of"], *rule["lineage"])
    _fail_if(
        frame,
        _any_null(required_values),
        f"{table_name} contains null grain/as-of/lineage values",
    )
    non_nullable_outputs = tuple(
        name
        for name, _, nullable in ML_OUTPUT_CONTRACTS[table_name]
        if not nullable and name not in required_values
    )
    if non_nullable_outputs:
        _fail_if(
            frame,
            _any_null(non_nullable_outputs),
            f"{table_name} contains null required output values",
        )
    floating_outputs = tuple(
        name
        for name, data_type, _ in ML_OUTPUT_CONTRACTS[table_name]
        if data_type == "double"
    )
    if floating_outputs:
        _fail_if(
            frame,
            _any_non_finite(floating_outputs),
            f"{table_name} contains NaN or infinite floating-point values",
        )
    _fail_if(
        frame,
        _any_blank(rule["lineage"]),
        f"{table_name} contains blank lineage values",
    )
    duplicates = frame.groupBy(*rule["grain"]).count()
    _fail_if(
        duplicates,
        F.col("count") != 1,
        f"{table_name} contains duplicate grain keys",
    )
    for probability in rule["probabilities"]:
        _fail_if(
            frame,
            F.col(probability).isNull()
            | (F.col(probability) < 0.0)
            | (F.col(probability) > 1.0),
            f"{table_name}.{probability} is null or outside [0, 1]",
        )
    TABLE_VALIDATORS[table_name](frame)
    return frame.count()


In [ ]:
validated_counts = {}
for required_table in ML_OUTPUT_CONTRACTS:
    validated_counts[required_table] = validate_required_table(required_table)

print("Required ML Reporting gate passed.")
for table_name, row_count in validated_counts.items():
    print(f"  {table_name}: {row_count:,} rows")
